# <center><b>Baseline Model Development and Comparison</b></center>

In [1]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv('../CSVs/08_model_ready_dataset.csv').drop(columns=['Unnamed: 0'])

In [3]:
# Changing columns names for more readability
df.columns = ['village_code', 'visit_month', 'spray_status', 'total_residents',
       'clinic_tests', 'total_rdts', 'avg_distance_to_health_facility_km',
       'incidence_per_1000', 'positivity_rate', 'dist_missing', 'rainfall',
       'rainfall_total', 'humidity', 'temp', 'max_temp', 'min_temp', 'temp_range',
       'prev_rainfall', 'prev2_rainfall', 'prev3_rainfall',
       'prev4_rainfall', 'prev_rainfall_total', 'prev2_rainfall_total',
       'prev3_rainfall_total', 'prev4_rainfall_total', 'prev_humidity',
       'prev2_humidity', 'prev3_humidity', 'prev4_humidity', 'prev_temp', 'prev2_temp',
       'prev3_temp', 'prev4_temp', 'prev_max_temp', 'prev2_max_temp',
       'prev3_max_temp', 'prev4_max_temp', 'prev_min_temp', 'prev2_min_temp',
       'prev3_min_temp', 'prev4_min_temp', 'prev_temp_range', 'prev2_temp_range',
       'prev3_temp_range', 'prev4_temp_range', 'prev_incidence_per_1000', 'mpi']

In [4]:
village_codes = df['village_code'].unique()
village_codes.sort()
split_point_village = int(len(village_codes) * .8)
village_bool = df['village_code'] <= village_codes[split_point_village]

In [5]:
dates = df['visit_month'].unique()
dates.sort()
split_point_date = int(len('visit_month') * .8)
month_bool = df['visit_month'] <= dates[split_point_date]

In [6]:
def chrono(chrono_bool):
    """
    Function for specifying which boolean to use for the (chronological or spatial) split
    """
    return month_bool if chrono_bool else village_bool

In [7]:
# 80% / 20% chronological split
target_columns = ['incidence_per_1000'] # These are the columns our model tries to predict
# And these are poor or useless columns that carry information about which villages have suffered epidemics before
# Causing it to overfit by predicting historical average
useless_columns = ['total_residents', 'visit_month', 'village_code', 'avg_distance_to_health_facility_km', 'clinic_tests',
                   'rainfall', 'rainfall_total', 'total_rdts', 'positivity_rate'] \
                + ['prev_rainfall', 'prev_rainfall_total', 'prev_min_temp', 'prev_max_temp', 'prev_temp', 'prev_humidity'] \
                + ['prev2_rainfall_total', 'prev3_rainfall_total', 'prev4_rainfall_total', 'prev_temp_range', 'prev2_temp_range',\
                   'prev3_temp_range', 'prev4_temp_range', 'temp_range', 'spray_status']
                 # + ['humidity', 'temp', 'max_temp', 'min_temp', 'temp_range', 'total_rdts']
# Train data becomes the majority (80%)
X_train = df[chrono(1)]
y_train = X_train[target_columns]
# Test data becomes the minority (20%)
X_test = df[~chrono(1)]
y_test = X_test[target_columns]
# We finally drop the target columns from the X data
X_train = X_train.drop(columns=target_columns + useless_columns) # We also don't need 'visit_month' anymore it isn't important in training
X_test = X_test.drop(columns=target_columns + useless_columns)

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
model, ridge = joblib.load('../Model Pkl Files/rf_model.pkl'), joblib.load('../Model Pkl Files/linear_model.pkl')